# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col


# Reading from Bronze Table

In [0]:
df = spark.table('de_project.bronze.crm_cust_info')
df.display()

# Exploring the Data

## Checking null value in each column

In [0]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

In [0]:
df.select('cst_gndr').distinct().show()
df.select('cst_marital_status').distinct().show()

# Silver Transformations

## Remove Records with Missing Customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("N/A")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("N/A")
    )
)

## Rename Columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Writing to Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("de_project.silver.crm_customers")

In [0]:
%sql
SELECT * FROM de_project.silver.crm_customers LIMIT 10